# 06 Baseline Retrieval

## Goal

This notebook evaluates the first retrieval layer of RiskRadar AI.

The workflow is:

```text
user question
→ embed question
→ search Chroma vector database
→ retrieve SEC filing chunks
→ inspect citations
→ compare filtered vs unfiltered search
→ score retrieval quality
```

This notebook does not generate final LLM answers yet.

The purpose is to test whether the retriever can find useful evidence before we add answer generation.

Good RAG starts with good retrieval.

In [1]:
# Import tools for file and folder paths
from pathlib import Path

# Import pandas for working with tables
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import ChromaDB for loading the local vector database
import chromadb

# Import SentenceTransformer for query embeddings
from sentence_transformers import SentenceTransformer

# Import textwrap for cleaner text previews
import textwrap

# Import time for retrieval timing
import time

In [2]:
# Detect the project root automatically
# If this notebook is inside the notebooks folder, move one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Create main project paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VECTORSTORE_DIR = DATA_DIR / "vectorstore"

# Set Chroma vector database folder
CHROMA_DIR = VECTORSTORE_DIR / "chroma_sec_10k"

# Print paths to confirm everything is correct
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Vectorstore folder:", VECTORSTORE_DIR)
print("Chroma folder:", CHROMA_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
Processed data folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
Vectorstore folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore
Chroma folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


#### Load RAG chunks

In [3]:
# Set path to final RAG chunks from notebook 04
rag_chunks_file = PROCESSED_DIR / "sec_10k_rag_chunks.csv"

# Check that the chunk file exists
if not rag_chunks_file.exists():
    raise FileNotFoundError(
        f"Could not find {rag_chunks_file}. Run 04_chunking_experiments.ipynb first."
    )

# Load RAG chunks
rag_chunks_df = pd.read_csv(rag_chunks_file)

# Preview chunk dataset
rag_chunks_df.head()

,chunk_id,document_id,ticker,company_name,filing_date,accession_number,filing_url,section_name,source_label,citation_label,chunk_index,chunk_size,chunk_overlap,start_word,end_word,chunk_word_count,chunk_character_count,chunk_text
0,AAPL_2025-10-31_item_1_business_chunk_0000,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",0,350,75,0,350,350,2125,Item 1. Business Company Background The Compan...
1,AAPL_2025-10-31_item_1_business_chunk_0001,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",1,350,75,275,625,350,2341,Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL_2025-10-31_item_1_business_chunk_0002,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2",2,350,75,550,900,350,2429,"Greater China includes China mainland, Hong Ko..."
3,AAPL_2025-10-31_item_1_business_chunk_0003,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",3,350,75,825,1175,350,2563,by imitating the Company’s products and infrin...
4,AAPL_2025-10-31_item_1_business_chunk_0004,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",4,350,75,1100,1450,350,2357,provide products and services at little or no ...


#### Load embedding summary

In [4]:
# Set path to embedding summary from notebook 05
embedding_summary_file = PROCESSED_DIR / "sec_10k_embedding_summary.csv"

# Check that the embedding summary exists
if not embedding_summary_file.exists():
    raise FileNotFoundError(
        f"Could not find {embedding_summary_file}. Run 05_embeddings_and_vector_store.ipynb first."
    )

# Load embedding summary
embedding_summary = pd.read_csv(embedding_summary_file)

# Display embedding summary
embedding_summary

,setting,value
0,embedding_model,sentence-transformers/all-MiniLM-L6-v2
1,total_chunks,520
2,embedding_dimensions,384
3,vectorstore,ChromaDB
4,collection_name,sec_10k_rag_chunks
5,chroma_path,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...


#### Extract vectorstore settings

In [5]:
# Convert embedding summary into a dictionary
embedding_settings = dict(
    zip(
        embedding_summary["setting"],
        embedding_summary["value"]
    )
)

# Get model name from notebook 05 settings
EMBEDDING_MODEL_NAME = embedding_settings["embedding_model"]

In [6]:
# Get collection name from notebook 05 settings
COLLECTION_NAME = embedding_settings["collection_name"]

# Display settings
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Collection name:", COLLECTION_NAME)
print("Chroma path:", CHROMA_DIR)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Collection name: sec_10k_rag_chunks
Chroma path: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


#### Load embedding model

In [7]:
# Start timer
start_time = time.time()

# Load the same embedding model used to build the vector store
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# End timer
end_time = time.time()

# Print loading time
print("Loaded embedding model:", EMBEDDING_MODEL_NAME)
print("Load time in seconds:", round(end_time - start_time, 2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
Load time in seconds: 2.01


#### Connect to Chroma collection

In [8]:
# Check that the Chroma folder exists
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find Chroma folder: {CHROMA_DIR}. Run 05_embeddings_and_vector_store.ipynb first."
    )

# Create persistent Chroma client
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# Load existing collection
collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

# Print collection count
print("Collection name:", COLLECTION_NAME)
print("Records in collection:", collection.count())

Collection name: sec_10k_rag_chunks
Records in collection: 520


#### Validate collection count

In [9]:
# Count rows in chunk CSV
chunk_count = len(rag_chunks_df)

# Count records in Chroma
collection_count = collection.count()

# Print both counts
print("Rows in chunk dataset:", chunk_count)
print("Records in Chroma collection:", collection_count)

Rows in chunk dataset: 520
Records in Chroma collection: 520


In [10]:
# Stop if counts do not match
if chunk_count != collection_count:
    raise ValueError("Chunk dataset count does not match Chroma collection count.")

# Confirm validation passed
print("Chunk dataset and Chroma collection match.")

Chunk dataset and Chroma collection match.


## Baseline Retrieval Design

The retrieval function will support:

```text
semantic query search
ticker filter
section filter
top-k results
citation metadata
retrieval timing
```

Important note:

```text
Lower Chroma distance means a closer match.
```

The output should show:

```text
rank
distance
ticker
company
filing date
section
citation label
filing URL
retrieved text
```

#### Build Chroma filter helper

In [11]:
def build_chroma_where_filter(ticker=None, section_name=None):
    """
    Build a Chroma-compatible metadata filter.

    Chroma requires exactly one top-level filter operator.
    If we use multiple conditions, we combine them with $and.
    """

    # Create a list to hold filter conditions
    filter_conditions = []

    # Add ticker filter if provided
    if ticker is not None:
        filter_conditions.append({"ticker": ticker})

    # Add section filter if provided
    if section_name is not None:
        filter_conditions.append({"section_name": section_name})

    # No filters
    if len(filter_conditions) == 0:
        return None

    # One filter can be returned directly
    if len(filter_conditions) == 1:
        return filter_conditions[0]

    # Multiple filters need Chroma's $and operator
    return {"$and": filter_conditions}

#### Test filter helper

In [12]:
# Test no filter
print("No filter:", build_chroma_where_filter())

# Test ticker-only filter
print("Ticker filter:", build_chroma_where_filter(ticker="TSLA"))

# Test ticker and section filter
print(
    "Ticker + section filter:",
    build_chroma_where_filter(
        ticker="TSLA",
        section_name="item_1a_risk_factors"
    )
)

No filter: None
Ticker filter: {'ticker': 'TSLA'}
Ticker + section filter: {'$and': [{'ticker': 'TSLA'}, {'section_name': 'item_1a_risk_factors'}]}


#### Main retrieval function

In [13]:
def search_sec_chunks(query, top_k=5, ticker=None, section_name=None):
    """
    Search SEC filing chunks using semantic similarity.

    Parameters:
    - query: user question
    - top_k: number of chunks to return
    - ticker: optional company ticker filter
    - section_name: optional SEC section filter
    """

    # Start timer
    start_time = time.time()

    # Embed the user query
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Build Chroma metadata filter
    where_filter = build_chroma_where_filter(
        ticker=ticker,
        section_name=section_name
    )

    # Query Chroma collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    # End timer
    end_time = time.time()

    # Create an empty list for result records
    result_records = []

    # Handle empty results safely
    if len(results["documents"][0]) == 0:
        return pd.DataFrame()

    # Loop through returned results
    for rank, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        # Store readable retrieval result
        result_records.append({
            "rank": rank,
            "distance": distance,
            "query": query,
            "ticker_filter": ticker,
            "section_filter": section_name,
            "ticker": metadata.get("ticker", ""),
            "company_name": metadata.get("company_name", ""),
            "filing_date": metadata.get("filing_date", ""),
            "section_name": metadata.get("section_name", ""),
            "citation_label": metadata.get("citation_label", ""),
            "filing_url": metadata.get("filing_url", ""),
            "chunk_text": doc,
            "retrieval_time_seconds": round(end_time - start_time, 4)
        })

    # Convert result records into a DataFrame
    results_df = pd.DataFrame(result_records)

    # Return retrieval results
    return results_df

#### Pretty result preview function

In [14]:
def preview_search_results(results_df, text_chars=500):
    """
    Display retrieval results in a readable way.
    """

    # Handle empty results
    if results_df.empty:
        print("No results found.")
        return

    # Loop through each result
    for _, row in results_df.iterrows():

        # Print result metadata
        print("=" * 100)
        print("Rank:", row["rank"])
        print("Distance:", row["distance"])
        print("Ticker:", row["ticker"])
        print("Company:", row["company_name"])
        print("Section:", row["section_name"])
        print("Citation:", row["citation_label"])
        print("Filing URL:", row["filing_url"])
        print("Retrieval time:", row["retrieval_time_seconds"], "seconds")
        print("-" * 100)

        # Wrap and print chunk preview
        preview_text = row["chunk_text"][:text_chars]
        print(textwrap.fill(preview_text, width=110))
        print()

#### Baseline search without filters

In [15]:
# Define a broad test query
query = "What risks does Tesla mention about supply chain and manufacturing?"

# Run unfiltered semantic search
unfiltered_results = search_sec_chunks(
    query=query,
    top_k=5
)

# Show compact result table
unfiltered_results[
    [
        "rank",
        "distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rank,distance,ticker,section_name,citation_label
0,1,0.895707,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
1,2,0.925222,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,0.972546,AAPL,item_1a_risk_factors,"AAPL 2025-10-31 10-K, item_1a_risk_factors, ch..."
3,4,0.987999,AMD,item_1a_risk_factors,"AMD 2026-02-04 10-K, item_1a_risk_factors, chu..."
4,5,0.990986,AAPL,item_1a_risk_factors,"AAPL 2025-10-31 10-K, item_1a_risk_factors, ch..."


#### Preview unfiltered results

In [16]:
# Preview retrieved evidence from unfiltered search
preview_search_results(
    results_df=unfiltered_results,
    text_chars=700
)

Rank: 1
Distance: 0.8957070112228394
Ticker: NVDA
Company: NVIDIA
Section: item_1a_risk_factors
Citation: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 4
Filing URL: https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
Retrieval time: 0.0311 seconds
----------------------------------------------------------------------------------------------------
site and the information on it or connected to it are not a part of this Annual Report on Form 10-K. Item 1A.
Risk Factors The following risk factors should be considered in addition to the other information in this
Annual Report on Form 10-K. The following risks could harm our business, financial condition, results of
operations or reputation, which could cause our stock price to decline. Additional risks, trends and
uncertainties not presently known to us or that we currently believe are immaterial may also harm our
business, financial condition, results of operations or reputation. Risk Factors Summary 

#### Search with ticker filter

In [17]:
# Run search with only Tesla ticker filter
ticker_filtered_results = search_sec_chunks(
    query=query,
    top_k=5,
    ticker="TSLA"
)

# Show compact result table
ticker_filtered_results[
    [
        "rank",
        "distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rank,distance,ticker,section_name,citation_label
0,1,0.925222,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,1.018959,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,1.019058,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,1.027786,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,1.028263,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


#### Search with ticker and section filter

In [18]:
# Run search with Tesla and Risk Factors filters
section_filtered_results = search_sec_chunks(
    query=query,
    top_k=5,
    ticker="TSLA",
    section_name="item_1a_risk_factors"
)

# Show compact result table
section_filtered_results[
    [
        "rank",
        "distance",
        "ticker",
        "section_name",
        "citation_label"
    ]
]

,rank,distance,ticker,section_name,citation_label
0,1,0.925222,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
1,2,1.018959,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,3,1.019058,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
3,4,1.027786,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
4,5,1.028263,TSLA,item_1a_risk_factors,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


#### Compare filtered vs unfiltered search

In [19]:
# Create comparison records
comparison_records = []

# Store top result from unfiltered search
comparison_records.append({
    "search_type": "unfiltered",
    "top_ticker": unfiltered_results.iloc[0]["ticker"],
    "top_section": unfiltered_results.iloc[0]["section_name"],
    "top_distance": unfiltered_results.iloc[0]["distance"],
    "top_citation": unfiltered_results.iloc[0]["citation_label"]
})

# Store top result from ticker-filtered search
comparison_records.append({
    "search_type": "ticker_filter",
    "top_ticker": ticker_filtered_results.iloc[0]["ticker"],
    "top_section": ticker_filtered_results.iloc[0]["section_name"],
    "top_distance": ticker_filtered_results.iloc[0]["distance"],
    "top_citation": ticker_filtered_results.iloc[0]["citation_label"]
})

# Store top result from ticker-and-section-filtered search
comparison_records.append({
    "search_type": "ticker_and_section_filter",
    "top_ticker": section_filtered_results.iloc[0]["ticker"],
    "top_section": section_filtered_results.iloc[0]["section_name"],
    "top_distance": section_filtered_results.iloc[0]["distance"],
    "top_citation": section_filtered_results.iloc[0]["citation_label"]
})

# Convert comparison to DataFrame
filter_comparison_df = pd.DataFrame(comparison_records)

# Display comparison
filter_comparison_df

,search_type,top_ticker,top_section,top_distance,top_citation
0,unfiltered,NVDA,item_1a_risk_factors,0.895707,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch..."
1,ticker_filter,TSLA,item_1a_risk_factors,0.925222,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."
2,ticker_and_section_filter,TSLA,item_1a_risk_factors,0.925222,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch..."


#### Retrieval evaluation plan

## Retrieval Evaluation

Now we create a small test set.

Each test question has:

```text
query
expected ticker
expected section
```

We will measure:

```text
ticker hit@k
section hit@k
ticker + section hit@k
top result distance
retrieval time
```

This is not a perfect evaluation yet, but it is a strong first version.

The goal is to prove that the retriever usually pulls evidence from the correct company and section.

#### Create retrieval test set

In [20]:
# Create baseline retrieval evaluation questions
# These questions only use companies currently loaded in the vector database
retrieval_test_set = pd.DataFrame([
    {
        "query": "What AI and competition risks does NVIDIA mention?",
        "expected_ticker": "NVDA",
        "expected_section": "item_1a_risk_factors"
    },
    {
        "query": "What cybersecurity risks does Microsoft mention?",
        "expected_ticker": "MSFT",
        "expected_section": "item_1a_risk_factors"
    },
    {
        "query": "What supply chain risks does Tesla mention?",
        "expected_ticker": "TSLA",
        "expected_section": "item_1a_risk_factors"
    },
    {
        "query": "What competition risks does Apple describe?",
        "expected_ticker": "AAPL",
        "expected_section": "item_1a_risk_factors"
    },
    {
        "query": "What semiconductor competition risks does AMD mention?",
        "expected_ticker": "AMD",
        "expected_section": "item_1a_risk_factors"
    },
    {
        "query": "What does Tesla say about results of operations?",
        "expected_ticker": "TSLA",
        "expected_section": "item_7_mda"
    },
    {
        "query": "What does Microsoft say about financial condition and results?",
        "expected_ticker": "MSFT",
        "expected_section": "item_7_mda"
    },
    {
        "query": "What does Apple say about products and business operations?",
        "expected_ticker": "AAPL",
        "expected_section": "item_1_business"
    }
])

# Display test set
retrieval_test_set

,query,expected_ticker,expected_section
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors
5,What does Tesla say about results of operations?,TSLA,item_7_mda
6,What does Microsoft say about financial condit...,MSFT,item_7_mda
7,What does Apple say about products and busines...,AAPL,item_1_business


#### Evaluate retrieval with filters

In [21]:
def evaluate_filtered_retrieval(test_set, top_k=5):
    """
    Evaluate retrieval using expected ticker and expected section as filters.

    This tests whether semantic search can find relevant chunks inside the right scope.
    """

    # Create empty list for evaluation records
    eval_records = []

    # Loop through each test case
    for _, test in test_set.iterrows():

        # Run retrieval with expected filters
        results_df = search_sec_chunks(
            query=test["query"],
            top_k=top_k,
            ticker=test["expected_ticker"],
            section_name=test["expected_section"]
        )

        # Handle no results
        if results_df.empty:
            eval_records.append({
                "query": test["query"],
                "expected_ticker": test["expected_ticker"],
                "expected_section": test["expected_section"],
                "num_results": 0,
                "top_ticker": None,
                "top_section": None,
                "top_distance": None,
                "ticker_hit_at_k": False,
                "section_hit_at_k": False,
                "pair_hit_at_k": False,
                "top_citation": None,
                "top_text_preview": None
            })
            continue

        # Check whether expected ticker appears in top-k results
        ticker_hit = test["expected_ticker"] in results_df["ticker"].tolist()

        # Check whether expected section appears in top-k results
        section_hit = test["expected_section"] in results_df["section_name"].tolist()

        # Check whether expected ticker and section appear together in top-k
        pair_hit = (
            (
                (results_df["ticker"] == test["expected_ticker"])
                & (results_df["section_name"] == test["expected_section"])
            )
            .any()
        )

        # Store evaluation result
        eval_records.append({
            "query": test["query"],
            "expected_ticker": test["expected_ticker"],
            "expected_section": test["expected_section"],
            "num_results": len(results_df),
            "top_ticker": results_df.iloc[0]["ticker"],
            "top_section": results_df.iloc[0]["section_name"],
            "top_distance": results_df.iloc[0]["distance"],
            "ticker_hit_at_k": ticker_hit,
            "section_hit_at_k": section_hit,
            "pair_hit_at_k": pair_hit,
            "top_citation": results_df.iloc[0]["citation_label"],
            "top_text_preview": results_df.iloc[0]["chunk_text"][:300]
        })

    # Convert evaluation records to DataFrame
    eval_df = pd.DataFrame(eval_records)

    # Return evaluation results
    return eval_df

#### Run filtered retrieval evaluation

In [22]:
# Run filtered retrieval evaluation
filtered_eval_df = evaluate_filtered_retrieval(
    test_set=retrieval_test_set,
    top_k=5
)

# Display evaluation results
filtered_eval_df

,query,expected_ticker,expected_section,num_results,top_ticker,top_section,top_distance,ticker_hit_at_k,section_hit_at_k,pair_hit_at_k,top_citation,top_text_preview
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,5,NVDA,item_1a_risk_factors,0.921209,True,True,True,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch...",may not achieve the desired results as designe...
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,5,MSFT,item_1a_risk_factors,0.861831,True,True,True,"MSFT 2025-07-30 10-K, item_1a_risk_factors, ch...",previously disclosed in our Form 8-K filed wit...
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,5,TSLA,item_1a_risk_factors,0.867302,True,True,True,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch...",be challenging due to our limited operating hi...
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,5,AAPL,item_1a_risk_factors,0.519304,True,True,True,"AAPL 2025-10-31 10-K, item_1a_risk_factors, ch...",their products to offer more competitive solut...
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,5,AMD,item_1a_risk_factors,0.877353,True,True,True,"AMD 2026-02-04 10-K, item_1a_risk_factors, chu...",which could materially adversely impact our bu...
5,What does Tesla say about results of operations?,TSLA,item_7_mda,3,TSLA,item_7_mda,1.572934,True,True,True,"TSLA 2026-01-29 10-K, item_7_mda, chunk 0","Item 7, Management's Discussion and Analysis o..."
6,What does Microsoft say about financial condit...,MSFT,item_7_mda,4,MSFT,item_7_mda,0.996882,True,True,True,"MSFT 2025-07-30 10-K, item_7_mda, chunk 0","Item 7, “Management’s Discussion and Analysis ..."
7,What does Apple say about products and busines...,AAPL,item_1_business,5,AAPL,item_1_business,0.694691,True,True,True,"AAPL 2025-10-31 10-K, item_1_business, chunk 6",rights around the world. No single intellectua...


#### Summarize filtered retrieval evaluation

In [23]:
# Calculate retrieval metrics
filtered_eval_summary = pd.DataFrame({
    "metric": [
        "test_questions",
        "ticker_hit_rate_at_5",
        "section_hit_rate_at_5",
        "pair_hit_rate_at_5",
        "average_top_distance"
    ],
    "value": [
        len(filtered_eval_df),
        filtered_eval_df["ticker_hit_at_k"].mean(),
        filtered_eval_df["section_hit_at_k"].mean(),
        filtered_eval_df["pair_hit_at_k"].mean(),
        filtered_eval_df["top_distance"].mean()
    ]
})

# Display summary
filtered_eval_summary

,metric,value
0,test_questions,8.000000
1,ticker_hit_rate_at_5,1.000000
2,section_hit_rate_at_5,1.000000
3,pair_hit_rate_at_5,1.000000
4,average_top_distance,0.913938


#### Evaluate retrieval without filters

In [24]:
def evaluate_unfiltered_retrieval(test_set, top_k=5):
    """
    Evaluate retrieval without metadata filters.

    This tests whether semantic search alone can find the correct company and section.
    """

    # Create empty list for evaluation records
    eval_records = []

    # Loop through each test case
    for _, test in test_set.iterrows():

        # Run unfiltered retrieval
        results_df = search_sec_chunks(
            query=test["query"],
            top_k=top_k
        )

        # Handle no results
        if results_df.empty:
            eval_records.append({
                "query": test["query"],
                "expected_ticker": test["expected_ticker"],
                "expected_section": test["expected_section"],
                "num_results": 0,
                "top_ticker": None,
                "top_section": None,
                "top_distance": None,
                "ticker_hit_at_k": False,
                "section_hit_at_k": False,
                "pair_hit_at_k": False,
                "top_citation": None,
                "top_text_preview": None
            })
            continue

        # Check whether expected ticker appears in top-k results
        ticker_hit = test["expected_ticker"] in results_df["ticker"].tolist()

        # Check whether expected section appears in top-k results
        section_hit = test["expected_section"] in results_df["section_name"].tolist()

        # Check whether expected ticker and section appear together in top-k
        pair_hit = (
            (
                (results_df["ticker"] == test["expected_ticker"])
                & (results_df["section_name"] == test["expected_section"])
            )
            .any()
        )

        # Store evaluation result
        eval_records.append({
            "query": test["query"],
            "expected_ticker": test["expected_ticker"],
            "expected_section": test["expected_section"],
            "num_results": len(results_df),
            "top_ticker": results_df.iloc[0]["ticker"],
            "top_section": results_df.iloc[0]["section_name"],
            "top_distance": results_df.iloc[0]["distance"],
            "ticker_hit_at_k": ticker_hit,
            "section_hit_at_k": section_hit,
            "pair_hit_at_k": pair_hit,
            "top_citation": results_df.iloc[0]["citation_label"],
            "top_text_preview": results_df.iloc[0]["chunk_text"][:300]
        })

    # Convert evaluation records to DataFrame
    eval_df = pd.DataFrame(eval_records)

    # Return evaluation results
    return eval_df

#### Run unfiltered retrieval evaluation

In [25]:
# Run unfiltered retrieval evaluation
unfiltered_eval_df = evaluate_unfiltered_retrieval(
    test_set=retrieval_test_set,
    top_k=5
)

# Display evaluation results
unfiltered_eval_df

,query,expected_ticker,expected_section,num_results,top_ticker,top_section,top_distance,ticker_hit_at_k,section_hit_at_k,pair_hit_at_k,top_citation,top_text_preview
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,5,NVDA,item_7_mda,0.764672,True,True,False,"NVDA 2026-02-25 10-K, item_7_mda, chunk 0",Item 7. Management's Discussion and Analysis o...
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,5,AMD,item_7_mda,0.765726,True,True,True,"AMD 2026-02-04 10-K, item_7_mda, chunk 103","services, and our broader enterprise IT enviro..."
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,5,TSLA,item_1a_risk_factors,0.867302,True,True,True,"TSLA 2026-01-29 10-K, item_1a_risk_factors, ch...",be challenging due to our limited operating hi...
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,5,AAPL,item_1a_risk_factors,0.519304,True,True,True,"AAPL 2025-10-31 10-K, item_1a_risk_factors, ch...",their products to offer more competitive solut...
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,5,AMD,item_1_business,0.870838,True,True,True,"AMD 2026-02-04 10-K, item_1_business, chunk 1","external financing on favorable terms, or at a..."
5,What does Tesla say about results of operations?,TSLA,item_7_mda,5,TSLA,item_1_business,0.957839,True,False,False,"TSLA 2026-01-29 10-K, item_1_business, chunk 21",enable our freedom to operate our innovations ...
6,What does Microsoft say about financial condit...,MSFT,item_7_mda,5,MSFT,item_7_mda,0.996882,True,True,True,"MSFT 2025-07-30 10-K, item_7_mda, chunk 0","Item 7, “Management’s Discussion and Analysis ..."
7,What does Apple say about products and busines...,AAPL,item_1_business,5,AAPL,item_1_business,0.694691,True,True,True,"AAPL 2025-10-31 10-K, item_1_business, chunk 6",rights around the world. No single intellectua...


#### Summarize unfiltered retrieval evaluation

In [26]:
# Calculate unfiltered retrieval metrics
unfiltered_eval_summary = pd.DataFrame({
    "metric": [
        "test_questions",
        "ticker_hit_rate_at_5",
        "section_hit_rate_at_5",
        "pair_hit_rate_at_5",
        "average_top_distance"
    ],
    "value": [
        len(unfiltered_eval_df),
        unfiltered_eval_df["ticker_hit_at_k"].mean(),
        unfiltered_eval_df["section_hit_at_k"].mean(),
        unfiltered_eval_df["pair_hit_at_k"].mean(),
        unfiltered_eval_df["top_distance"].mean()
    ]
})

# Display summary
unfiltered_eval_summary

,metric,value
0,test_questions,8.000000
1,ticker_hit_rate_at_5,1.000000
2,section_hit_rate_at_5,0.875000
3,pair_hit_rate_at_5,0.750000
4,average_top_distance,0.804657


#### Compare filtered vs unfiltered evaluation

In [27]:
# Add method labels
filtered_summary_labeled = filtered_eval_summary.copy()
filtered_summary_labeled["method"] = "filtered"

unfiltered_summary_labeled = unfiltered_eval_summary.copy()
unfiltered_summary_labeled["method"] = "unfiltered"

# Combine summaries
retrieval_summary_comparison = pd.concat(
    [
        filtered_summary_labeled,
        unfiltered_summary_labeled
    ],
    ignore_index=True
)

# Reorder columns
retrieval_summary_comparison = retrieval_summary_comparison[
    [
        "method",
        "metric",
        "value"
    ]
]

# Display comparison
retrieval_summary_comparison

,method,metric,value
0,filtered,test_questions,8.000000
1,filtered,ticker_hit_rate_at_5,1.000000
2,filtered,section_hit_rate_at_5,1.000000
3,filtered,pair_hit_rate_at_5,1.000000
4,filtered,average_top_distance,0.913938
5,unfiltered,test_questions,8.000000
6,unfiltered,ticker_hit_rate_at_5,1.000000
7,unfiltered,section_hit_rate_at_5,0.875000
8,unfiltered,pair_hit_rate_at_5,0.750000
9,unfiltered,average_top_distance,0.804657


#### Save retrieval evaluation outputs

In [28]:
# Set output paths
filtered_eval_file = PROCESSED_DIR / "sec_10k_filtered_retrieval_eval.csv"
unfiltered_eval_file = PROCESSED_DIR / "sec_10k_unfiltered_retrieval_eval.csv"
retrieval_summary_file = PROCESSED_DIR / "sec_10k_retrieval_eval_summary.csv"
filter_comparison_file = PROCESSED_DIR / "sec_10k_filter_comparison.csv"

# Save evaluation files
filtered_eval_df.to_csv(filtered_eval_file, index=False)
unfiltered_eval_df.to_csv(unfiltered_eval_file, index=False)
retrieval_summary_comparison.to_csv(retrieval_summary_file, index=False)
filter_comparison_df.to_csv(filter_comparison_file, index=False)

# Confirm files were saved
print("Saved filtered eval to:", filtered_eval_file)
print("Saved unfiltered eval to:", unfiltered_eval_file)
print("Saved retrieval summary to:", retrieval_summary_file)
print("Saved filter comparison to:", filter_comparison_file)

Saved filtered eval to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_filtered_retrieval_eval.csv
Saved unfiltered eval to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_unfiltered_retrieval_eval.csv
Saved retrieval summary to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_retrieval_eval_summary.csv
Saved filter comparison to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_filter_comparison.csv


In [29]:
# Create final checkpoint table
retrieval_checkpoint = pd.DataFrame({
    "output": [
        "Filtered retrieval evaluation",
        "Unfiltered retrieval evaluation",
        "Retrieval summary comparison",
        "Filter comparison"
    ],
    "path": [
        str(filtered_eval_file),
        str(unfiltered_eval_file),
        str(retrieval_summary_file),
        str(filter_comparison_file)
    ],
    "exists": [
        filtered_eval_file.exists(),
        unfiltered_eval_file.exists(),
        retrieval_summary_file.exists(),
        filter_comparison_file.exists()
    ]
})

# Display checkpoint table
retrieval_checkpoint

,output,path,exists
0,Filtered retrieval evaluation,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,Unfiltered retrieval evaluation,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Retrieval summary comparison,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
3,Filter comparison,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


In [30]:
# Validate that every expected ticker in the test set exists in the chunk dataset

# Get tickers available in the vector database source data
available_tickers = set(rag_chunks_df["ticker"].unique())

# Get expected tickers from the retrieval test set
expected_tickers = set(retrieval_test_set["expected_ticker"].unique())

# Find test tickers that are missing from the vector database
missing_test_tickers = expected_tickers - available_tickers

# Stop if any test ticker is missing
if missing_test_tickers:
    raise ValueError(
        f"These test tickers are not in the vector database: {missing_test_tickers}"
    )

# Confirm test set is valid
print("All retrieval test tickers exist in the vector database.")

All retrieval test tickers exist in the vector database.


## Baseline Retrieval Conclusion

This notebook tested the first retrieval layer of RiskRadar AI.

The project now has:

```text
user question
→ query embedding
→ Chroma semantic search
→ ticker filtering
→ section filtering
→ citation-ready retrieved evidence
→ retrieval evaluation
```

Key lesson:

```text
Filtered retrieval is usually better when the user already knows the company or filing section.
Unfiltered retrieval is useful for open-ended discovery across companies.
```

This notebook proves that the system can retrieve real SEC evidence before generating answers.

The next notebook will improve retrieval quality with hybrid search.

```text
semantic vector search
+ keyword search
+ result merging
+ better evidence ranking
```